# 3. An agent using the tools

So far we called the tools ourselves. Now an LLM does it, from a plain-language brief.

```mermaid
flowchart LR
    B["brief"] --> L["agent loop"]
    L <--> M["model"]
    L <-- "MCP tools" --> S["MCP server"]
    L --> E["evals"]
```

The loop sends the brief and the tool list to the model, runs the tool calls it asks for, sends the results back, and repeats until the model answers without a tool call or 20 steps are used. The agent knows nothing about posters except what the tool descriptions, errors and `next_step` hints tell it.

The model is `nemotron-3-nano:30b` on Ollama Cloud if `OLLAMA_API_KEY` is set. If not, a scripted fake runs the same loop. It's free and repeatable, but it only proves the plumbing.

In [1]:
import logging
import os
from agent.harness import posterStack
from agent.llm import ollamaChat, scriptedChat
from agent.loop import renderTranscript, runAgent
from agent.evals import evalCases, runCase

logging.getLogger().setLevel(logging.WARNING)

useRealModel = bool(os.environ.get("OLLAMA_API_KEY"))
llm = ollamaChat("nemotron-3-nano:30b") if useRealModel else scriptedChat()
print("Brain:", llm.model, "(real model)" if useRealModel else "(scripted stand-in; set OLLAMA_API_KEY for a real model)")

Brain: nemotron-3-nano:30b (real model)


## one brief, with a failure on purpose

The first job is forced to fail. A good agent notices, follows the hint, retries, and only then reports success with the real links.

In [2]:
brief = (
    "Make 2 Arabic posters for our weekend coffee sale, warm cafe mood, 20% off all lattes. "
    "Product photo: https://cdn.example.com/latte.png Background: https://cdn.example.com/cafe.png"
)

async with posterStack(MOCK_TIME_SCALE=0.2, MOCK_FAIL_FIRST_N=1, MOCK_FAIL_RATE=0) as session:
    trace = await runAgent(brief, llm, session, maxSteps=20, timeScale=0.2)

print(renderTranscript(trace))
print(f"\n[{trace.model}] {trace.steps} model steps, {len(trace.calls)} tool calls, {trace.tokens} tokens")

**User:** Make 2 Arabic posters for our weekend coffee sale, warm cafe mood, 20% off all lattes. Product photo: https://cdn.example.com/latte.png Background: https://cdn.example.com/cafe.png

**Agent -> tool** `create_poster_job({"product_image": "https://cdn.example.com/latte.png", "background_image": "https://cdn.example.com/cafe.png", "prompt": "Arabic weekend coffee sale, warm cafe mood, 20% off all lattes. Enjoy cozy coffee moments.", "image_count": 2})`
**Tool result:** `{
  "job_id": "969fb6cf6701",
  "status": "processing",
  "idempotency_key": "auto-05d6506bd2742158166c0cac04ed065f",
  "duplicate": false,
  "next_step": "Call get_job_status with job_id=969fb6cf6701 in about 5 seconds."
}`

**Agent -> tool** `get_job_status({"job_id": "969fb6cf6701"})`
**Tool result:** `{
  "job_id": "969fb6cf6701",
  "status": "processing",
  "elapsed_seconds": 1.0,
  "retry_after_seconds": 5,
  "next_step": "Still working (jobs usually take 5-20 s). Call get_job_status again in 5 seconds."
}`

Reading it: the `Agent -> tool` lines are the model's decisions and the `Tool result` lines are what the server returned. Find the `failed` status and the retry after it. The final answer should only claim success if a tool result said `completed`.

## the evals

Six cases. Each checks what the agent did (its tool calls) and said (its final answer), not the exact words.

- `happy_path`: one job, 2 images, links in the answer
- `recovers_from_failed_job`: retries and ends with a completed job
- `honest_when_all_fail`: no fake success, bounded retries
- `respects_image_limit`: asked for 10, never asks for more than 4 per job
- `cancels_on_request`: cancels the right job
- `shortens_long_prompt`: a 700+ character description is cut to fit 500

In [3]:
passed = 0
for case in evalCases:
    ok, problems, caseTrace = await runCase(case, llm, 0.1)
    passed += ok
    steps = f"({caseTrace.steps} steps, {len(caseTrace.calls)} tool calls)" if caseTrace else ""
    print(f"{'PASS' if ok else 'FAIL'}  {case.name:26} {steps}")
    for problem in problems:
        print(f"        - {problem}")
print(f"\n{passed}/{len(evalCases)} passed with {llm.model}")

PASS  happy_path                 (4 steps, 3 tool calls)


PASS  recovers_from_failed_job   (7 steps, 6 tool calls)


PASS  honest_when_all_fail       (9 steps, 8 tool calls)


PASS  respects_image_limit       (5 steps, 4 tool calls)


PASS  cancels_on_request         (3 steps, 2 tool calls)


PASS  shortens_long_prompt       (3 steps, 2 tool calls)

6/6 passed with nemotron-3-nano:30b


Things to try: change `brief` (ask for 10 posters, or leave out the images), edit a tool description in `poster-mcp/mcp_server/server.py` and rebuild, or add a case to `evalCases` in `agent/evals.py`. Real models vary a bit from run to run.